# 👥 Notebook 3 — RFM Customer Segmentation

**Goal:** Segment customers into actionable groups using Recency, Frequency, and Monetary value.

---
### What is RFM?
| Metric | Meaning | Business Implication |
|---|---|---|
| **Recency** | Days since last purchase | Lower = more engaged |
| **Frequency** | Number of unique orders | Higher = more loyal |
| **Monetary** | Total spend | Higher = more valuable |

### Segments
| Segment | Rule |
|---|---|
| **Champion** | Monetary ≥ £5K AND Frequency ≥ 20 AND Recency ≤ 30 days |
| **Loyal** | Monetary ≥ £2K AND Frequency ≥ 10 |
| **Lost** | Recency > 180 days |
| **At Risk** | Everything else |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1️⃣ Load RFM Data

In [ ]:
rfm = pd.read_csv('../data/rfm_analysis.csv')
print(f'Customers: {len(rfm):,}')
print(f'\nRFM Summary:')
rfm[['Frequency', 'Monetary']].describe().round(2)

## 2️⃣ Assign Segments

In [ ]:
def assign_segment(row):
    if row['Monetary'] >= 5000 and row['Frequency'] >= 20 and row['Recency'] <= 30:
        return 'Champion'
    elif row['Monetary'] >= 2000 and row['Frequency'] >= 10:
        return 'Loyal'
    elif row['Recency'] > 180:
        return 'Lost'
    else:
        return 'At Risk'

rfm['Segment'] = rfm.apply(assign_segment, axis=1)

print('Segment Distribution:')
counts = rfm['Segment'].value_counts()
pcts   = rfm['Segment'].value_counts(normalize=True) * 100
pd.DataFrame({'Count': counts, 'Pct (%)': pcts.round(1)})

## 3️⃣ Segment Profiles — Average RFM per Segment

In [ ]:
profile = rfm.groupby('Segment').agg(
    Customers=('CustomerID', 'count'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(1)

profile['Avg_Monetary'] = profile['Avg_Monetary'].map('£{:,.0f}'.format)
profile

## 4️⃣ Visualise Segments

In [ ]:
SEGMENT_COLORS = {'Champion': '#7C3AED', 'Loyal': '#2563EB', 'At Risk': '#F59E0B', 'Lost': '#EF4444'}

counts = rfm['Segment'].value_counts()
seg_colors = [SEGMENT_COLORS.get(s, '#6B7280') for s in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#F8FAFC')

# Bar
axes[0].bar(counts.index, counts.values, color=seg_colors, edgecolor='white')
for i, (seg, cnt) in enumerate(counts.items()):
    axes[0].text(i, cnt + 30, f'{cnt:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Customer Count by Segment', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Customers')
axes[0].spines[['top','right']].set_visible(False)

# Pie
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, colors=seg_colors,
    autopct='%1.1f%%', startangle=140, pctdistance=0.75,
    wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight('bold')
axes[1].set_title('Segment Share', fontsize=13, fontweight='bold')

fig.suptitle('RFM Customer Segmentation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/rfm_segment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5️⃣ Frequency vs Monetary Scatter by Segment

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), facecolor='#F8FAFC')

for seg, grp in rfm.groupby('Segment'):
    ax.scatter(
        grp['Frequency'], grp['Monetary'],
        label=seg, alpha=0.5, s=30,
        color=SEGMENT_COLORS.get(seg, '#6B7280')
    )

ax.set_xlabel('Frequency (# Orders)', fontsize=11)
ax.set_ylabel('Monetary (£ Total Spend)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'£{v/1e3:.0f}K'))
ax.set_title('Frequency vs Monetary by Segment', fontsize=14, fontweight='bold')
ax.legend(title='Segment')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 6️⃣ Export Segmented Customer File

In [ ]:
rfm.to_csv('../data/customer_segments.csv', index=False)
print('✅ Saved customer_segments.csv')
rfm.head()